
# Gold Layer Notebook - Analisis Emisi SDG 12 (2014–2023)

**Deskripsi:**  
Notebook ini mengimplementasikan Gold Layer pipeline berbasis Medallion Architecture. Dataset yang digunakan adalah Silver Layer hasil pembersihan data EPA GHGRP 2014–2023.  
Tujuan:
- Melakukan clustering fasilitas industri berdasarkan emisi (CO2, CH4, N2O).  
- Membandingkan metode K-Means dan DBSCAN.  
- Menyimpan output Gold siap visualisasi.

**Kaitan dengan SDG 12:** Memantau dan mengelompokkan fasilitas industri untuk mendukung produksi bertanggung jawab dan pengurangan emisi gas rumah kaca.


In [ ]:

# Library
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import DBSCAN


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import os

# Spark session dengan konfigurasi optimal
spark = SparkSession.builder \
    .appName("GHGRP_Gold") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark session siap")


Mounted at /content/drive
Spark session siap


In [ ]:

# Load Silver Layer Parquet
SILVER_PATH = "/content/drive/MyDrive/ABD/data/silver/ghgrp_silver.parquet"
df_silver = spark.read.parquet(SILVER_PATH)
print(f"Total baris: {df_silver.count():,}")
print(f"Kolom yang tersedia: {df_silver.columns}")
df_silver.show(5)


Total baris: 66,568
Kolom yang tersedia: ['facility_id', 'facility_name', 'city', 'state', 'naics_code', 'sector', 'total_emissions', 'co2_emissions', 'ch4_emissions', 'n2o_emissions', 'sector_clean', 'log_emissions', 'year']
+-----------+--------------------+-----------+-----+----------+------------+---------------+-------------+-------------+-------------+------------+------------------+----+
|facility_id|       facility_name|       city|state|naics_code|      sector|total_emissions|co2_emissions|ch4_emissions|n2o_emissions|sector_clean|     log_emissions|year|
+-----------+--------------------+-----------+-----+----------+------------+---------------+-------------+-------------+-------------+------------+------------------+----+
|    1000001|PSE Ferndale Gene...|   FERNDALE|   WA|  221112.0|Power Plants|     333193.564|     332854.9|        154.5|      184.164|Power Plants|12.716481874615932|2014|
|    1000002|Ardagh Glass Inc....|    DUNKIRK|   IN|  327213.0|    Minerals|     11495

In [ ]:

# Agregasi rata-rata emisi per fasilitas
df_features = df_silver.groupBy("facility_id").agg(
    F.mean("co2_emissions").alias("co2_mean"),
    F.mean("ch4_emissions").alias("ch4_mean"),
    F.mean("n2o_emissions").alias("n2o_mean")
)

# Convert Spark DF to Pandas
df_features_pd = df_features.toPandas()
X = df_features_pd[["co2_mean", "ch4_mean", "n2o_mean"]].values
print("Sample data (5 fasilitas pertama):")
print(X[:5])
print(f"\nShape: {X.shape}")


Sample data (5 fasilitas pertama):
[[5.82322889e+04 2.75833333e+01 3.41375556e+01]
 [5.48185000e+04 3.93512500e+03 3.07834000e+01]
 [4.12012880e+06 1.07342750e+04 1.84657488e+04]
 [7.56519650e+05 3.60352500e+02 4.50039600e+02]
 [1.75637248e+06 4.02565000e+03 6.88374040e+03]]

Shape: (8402, 3)


In [ ]:

# Feature scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Shape setelah scaling: {X_scaled.shape}")
print("Mean ~ 0, Std ~ 1:", X_scaled.mean(axis=0).round(2), X_scaled.std(axis=0).round(2))


Shape setelah scaling: (8402, 3)
Mean ~ 0, Std ~ 1: [-0. -0.  0.] [1. 1. 1.]


In [ ]:

# K-Means dengan PySpark ML
from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler
from pyspark.ml.clustering import KMeans as SparkKMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Siapkan features vector di Spark
assembler = VectorAssembler(inputCols=["co2_mean", "ch4_mean", "n2o_mean"], outputCol="features_raw")
df_features_vector = assembler.transform(df_features)

# Standard scaling di Spark
scaler_spark = SparkScaler(inputCol="features_raw", outputCol="features_scaled",
                           withStd=True, withMean=True)
scaler_model = scaler_spark.fit(df_features_vector)
df_scaled = scaler_model.transform(df_features_vector)

# Elbow method untuk cari k optimal
evaluator_silhouette = ClusteringEvaluator(featuresCol="features_scaled", metricName="silhouette")
silhouette_scores = []

for k in range(2, 7):
    kmeans = SparkKMeans(featuresCol="features_scaled", k=k, seed=42)
    model = kmeans.fit(df_scaled)
    predictions = model.transform(df_scaled)
    score = evaluator_silhouette.evaluate(predictions)
    silhouette_scores.append((k, score))
    print(f"k={k}: Silhouette = {score:.4f}")

# Pilih k terbaik (silhouette tertinggi)
best_k = max(silhouette_scores, key=lambda x: x[1])[0]
print(f"\n✅ k optimal: {best_k} dengan silhouette = {max(silhouette_scores, key=lambda x: x[1])[1]:.4f}")

# Fit final model dengan k optimal
final_kmeans = SparkKMeans(featuresCol="features_scaled", k=best_k, seed=42)
kmeans_model = final_kmeans.fit(df_scaled)
df_clustered = kmeans_model.transform(df_scaled)

# Ambil label cluster
df_features_pd = df_clustered.select("facility_id", "co2_mean", "ch4_mean", "n2o_mean", "prediction") \
                              .toPandas()
df_features_pd = df_features_pd.rename(columns={"prediction": "kmeans_cluster"})


k=2: Silhouette = 0.9486
k=3: Silhouette = 0.9569
k=4: Silhouette = 0.8581
k=5: Silhouette = 0.9305
k=6: Silhouette = 0.9364

✅ k optimal: 3 dengan silhouette = 0.9569


In [ ]:

# DDBSCAN dengan scikit-learn
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Gunakan X_scaled dari cell sebelumnya (sudah di-scale)
X_scaled_np = X_scaled  # ambil dari cell 4

# Cari eps optimal pakai k-distance graph
nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled_np)
distances, indices = nbrs.kneighbors(X_scaled_np)
distances = np.sort(distances[:, -1])  # jarak ke tetangga ke-5 (index 4 → -1)

# Ambil eps pada percentile 95 (elbow point)
eps_optimal = distances[int(0.95 * len(distances))]
print(f"DBSCAN eps optimal (95th percentile): {eps_optimal:.4f}")

# Run DBSCAN
dbscan = DBSCAN(eps=eps_optimal, min_samples=5)
db_labels = dbscan.fit_predict(X_scaled_np)

# Evaluasi DBSCAN (hanya untuk non-noise points)
mask = db_labels != -1
n_clusters_db = len(set(db_labels[mask]))
n_noise = (~mask).sum()

print(f"Jumlah cluster DBSCAN: {n_clusters_db}")
print(f"Jumlah noise points: {n_noise} ({n_noise/len(db_labels)*100:.1f}%)")

if n_clusters_db >= 2:
    db_sil = silhouette_score(X_scaled_np[mask], db_labels[mask])
    db_dbi = davies_bouldin_score(X_scaled_np[mask], db_labels[mask])
    print(f"Silhouette Score (non-noise): {db_sil:.4f}")
    print(f"Davies-Bouldin Index: {db_dbi:.4f}")
else:
    print("⚠️ DBSCAN hanya menghasilkan 1 cluster → tidak bisa dihitung silhouette")

df_features_pd["dbscan_cluster"] = db_labels


DBSCAN eps optimal (95th percentile): 0.0659
Jumlah cluster DBSCAN: 13
Jumlah noise points: 362 (4.3%)
Silhouette Score (non-noise): 0.1018
Davies-Bouldin Index: 0.4532


In [ ]:

# statistik cluster
print("\n=== STATISTIK CLUSTER K-MEANS ===")
cluster_stats = df_features_pd.groupby("kmeans_cluster")[["co2_mean", "ch4_mean", "n2o_mean"]].mean()
print(cluster_stats.round(2))

print("\n=== DISTRIBUSI CLUSTER K-MEANS ===")
print(df_features_pd["kmeans_cluster"].value_counts().sort_index())

print("\n=== STATISTIK CLUSTER DBSCAN ===")
if n_clusters_db > 0:
    db_stats = df_features_pd[df_features_pd["dbscan_cluster"] != -1] \
                         .groupby("dbscan_cluster")[["co2_mean", "ch4_mean", "n2o_mean"]].mean()
    print(db_stats.round(2))



=== STATISTIK CLUSTER K-MEANS ===
                  co2_mean    ch4_mean  n2o_mean
kmeans_cluster                                  
0                192706.95    15525.82   1018.51
1               6279298.16    17314.37  89691.52
2                 28489.50  1380234.99     36.71

=== DISTRIBUSI CLUSTER K-MEANS ===
kmeans_cluster
0    8200
1     184
2      18
Name: count, dtype: int64

=== STATISTIK CLUSTER DBSCAN ===
                  co2_mean   ch4_mean  n2o_mean
dbscan_cluster                                 
0                161842.78   12261.87    349.44
1               4718786.93   12967.76  22408.79
2                 36139.86  243921.63     55.33
3               3176929.55    1102.08   1331.87
4               2459649.07    6350.71  11448.71
5                 23852.69  315744.33     34.39
6               2756465.33    4455.20  13144.29
7               2925702.14    1535.90   2301.85
8               2173651.32    6238.90  10817.94
9               3431097.06    8713.56  16950.72
10 

In [ ]:

# simpan output Gold Layer
# Path output
GOLD_PATH = "/content/drive/MyDrive/ABD/data/gold/ghgrp_gold.parquet"
os.makedirs("/content/drive/MyDrive/ABD/data/gold", exist_ok=True)

df_features_pd.to_parquet(GOLD_PATH, index=False)
print(f"✅ Gold Layer tersimpan di: {GOLD_PATH}")
print(f"   Total fasilitas: {len(df_features_pd):,}")
print(f"   Kolom: {list(df_features_pd.columns)}")

# Juga simpan sebagai CSV untuk referensi
csv_path = "/content/drive/MyDrive/ABD/data/gold/ghgrp_gold.csv"
df_features_pd.to_csv(csv_path, index=False)
print(f"   CSV backup: {csv_path}")

spark.stop()
print("\n🎉 Gold Layer pipeline selesai!")


✅ Gold Layer tersimpan di: /content/drive/MyDrive/ABD/data/gold/ghgrp_gold.parquet
   Total fasilitas: 8,402
   Kolom: ['facility_id', 'co2_mean', 'ch4_mean', 'n2o_mean', 'kmeans_cluster', 'dbscan_cluster']
   CSV backup: /content/drive/MyDrive/ABD/data/gold/ghgrp_gold.csv

🎉 Gold Layer pipeline selesai!
